# Vision Transformer Layer Approximation Analysis
## Investigating Linear CKA and Block Skipping Effectiveness

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 8)

# Load data
results = pd.read_csv('/Users/mabelwylie/Documents/toast-extensions/results/results_window_grid_combined.csv')
accuracies = pd.read_csv('/Users/mabelwylie/Documents/toast-extensions/skipping_heads/accuracies/accuracies_window_grid_combined.csv')

print(f"Results shape: {results.shape}")
print(f"Accuracies shape: {accuracies.shape}")
print(f"\nUnique models: {results['model'].nunique()}")
print(f"Unique datasets: {results['dataset'].nunique()}")
print(f"Unique translators: {results['translator'].nunique()}")

## Data Preprocessing

In [ ]:
# Extract model short name
results['model_short'] = results['model'].str.split('/').str[-1]

# Extract number of approximated layers from approx_layer
def parse_approx_layers(s):
    if s == '[]' or pd.isna(s):
        return 0
    try:
        # Parse string representation of list
        import ast
        return len(ast.literal_eval(str(s)))
    except:
        return 0

results['num_approx_layers'] = results['approx_layer'].apply(parse_approx_layers)

print("Processed data:")
print(results[['model_short', 'dataset', 'translator', 'num_approx_layers', 'accuracy', 'params_saved_pct']].head(10))

## Best Approximations by Model and Dataset

In [ ]:
# Filter only linear approximations (not identity baseline)
linear_results = results[results['translator'] == 'linear'].copy()

# Sort by model, dataset, and accuracy (descending)
best_approximations = linear_results.sort_values(
    by=['model_short', 'dataset', 'accuracy'], 
    ascending=[True, True, False]
)

print(f"Total linear approximation experiments: {len(linear_results)}")
print(f"\nTop 20 best approximations:")
print(best_approximations[[
    'model_short', 'dataset', 'seed', 'approx_layer', 'num_approx_layers',
    'accuracy', 'original_accuracy', 'delta_acc', 'params_saved_pct'
]].head(20))

## Accuracy Summary by Model and Dataset

In [ ]:
# Group by model, dataset, and number of approximated layers
summary = linear_results.groupby(['model_short', 'dataset', 'num_approx_layers']).agg({
    'accuracy': ['mean', 'std', 'min', 'max'],
    'delta_acc': ['mean', 'max'],
    'params_saved_pct': 'first',
    'original_accuracy': 'mean'
}).reset_index()

summary.columns = ['model_short', 'dataset', 'num_layers', 'accuracy_mean', 'accuracy_std', 
                   'accuracy_min', 'accuracy_max', 'delta_mean', 'delta_max', 
                   'params_saved_pct', 'baseline_accuracy']

# Calculate relative accuracy retention
summary['accuracy_retention'] = (summary['accuracy_mean'] / summary['baseline_accuracy'] * 100).round(2)

print("Summary statistics:")
print(summary[['model_short', 'dataset', 'num_layers', 'accuracy_mean', 'accuracy_retention', 'params_saved_pct']].head(30))

## Sweet Spot Analysis: Best Accuracy-Compression Trade-offs

In [ ]:
# For each model-dataset pair, find the configuration with best accuracy
best_per_config = linear_results.groupby(['model_short', 'dataset']).apply(
    lambda x: x.loc[x['accuracy'].idxmax()]
).reset_index(drop=True)

# Also find configurations that maintain >95% relative accuracy with maximum compression
def find_sweet_spot(group):
    baseline = group['original_accuracy'].iloc[0]
    high_retention = group[group['accuracy'] / baseline >= 0.95].copy()
    
    if len(high_retention) > 0:
        # Find max compression while maintaining 95% retention
        return high_retention.loc[high_retention['params_saved_pct'].idxmax()]
    else:
        return group.iloc[0]

sweet_spots = linear_results.groupby(['model_short', 'dataset']).apply(find_sweet_spot).reset_index(drop=True)

print("\nSWEET SPOT CONFIGURATIONS (95%+ accuracy retention):")
print("="*100)
print(sweet_spots[[
    'model_short', 'dataset', 'approx_layer', 'accuracy', 'original_accuracy',
    'params_saved_pct', 'num_approx_layers'
]].to_string())

## Visualization: Accuracy vs Parameters Saved

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

models = sorted(linear_results['model_short'].unique())

for idx, model in enumerate(models):
    ax = axes[idx]
    model_data = linear_results[linear_results['model_short'] == model]
    
    # Plot each dataset with different color
    for dataset in sorted(model_data['dataset'].unique()):
        dataset_data = model_data[model_data['dataset'] == dataset]
        
        # Group by seed and num_approx_layers
        plot_data = dataset_data.groupby('num_approx_layers').agg({
            'accuracy': 'mean',
            'params_saved_pct': 'first'
        }).reset_index()
        
        ax.plot(plot_data['params_saved_pct'], plot_data['accuracy'], 
               marker='o', label=dataset, linewidth=2, markersize=6)
    
    # Add baseline
    baseline = linear_results[(linear_results['model_short'] == model) & 
                              (linear_results['translator'] == 'identity')]['accuracy'].mean()
    ax.axhline(baseline, color='gray', linestyle='--', alpha=0.5, label='Baseline')
    
    ax.set_xlabel('Parameters Saved (%)', fontsize=10)
    ax.set_ylabel('Accuracy', fontsize=10)
    ax.set_title(model, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/Users/mabelwylie/Documents/toast-extensions/accuracy_vs_params.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to: accuracy_vs_params.png")

## Accuracy Loss Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Group by dataset and show accuracy loss distribution
for idx, dataset in enumerate(sorted(linear_results['dataset'].unique())):
    ax = axes[idx]
    dataset_data = linear_results[linear_results['dataset'] == dataset]
    
    # Box plot by model
    data_for_box = [group['delta_acc'].values 
                    for name, group in dataset_data.groupby('model_short')]
    models_list = [name for name, _ in dataset_data.groupby('model_short')]
    
    bp = ax.boxplot(data_for_box, labels=models_list, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
    
    ax.set_ylabel('Accuracy Loss (delta_acc)', fontsize=10)
    ax.set_title(f'Dataset: {dataset}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('/Users/mabelwylie/Documents/toast-extensions/accuracy_loss_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to: accuracy_loss_distribution.png")

## Linear CKA Insights

Based on your observation that linear layers have similar linear CKA values, let's investigate which approximations work best when layers are similar.

In [ ]:
# Analyze the relationship between number of approximated layers and accuracy degradation
layer_impact = linear_results.groupby('num_approx_layers').agg({
    'accuracy': ['mean', 'std'],
    'delta_acc': ['mean', 'max'],
    'params_saved_pct': 'mean',
    'model_short': 'count'  # Count of experiments
}).reset_index()

layer_impact.columns = ['num_layers', 'mean_acc', 'std_acc', 'mean_loss', 'max_loss', 'mean_params', 'count']

print("\nIMPACT OF NUMBER OF APPROXIMATED LAYERS:")
print("="*80)
print(layer_impact.to_string(index=False))

# Plot the trend
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy retention
ax1.plot(layer_impact['num_layers'], layer_impact['mean_acc'], 
        marker='o', linewidth=2, markersize=8, color='steelblue')
ax1.fill_between(layer_impact['num_layers'], 
                 layer_impact['mean_acc'] - layer_impact['std_acc'],
                 layer_impact['mean_acc'] + layer_impact['std_acc'],
                 alpha=0.3, color='steelblue')
ax1.set_xlabel('Number of Approximated Layers', fontsize=11)
ax1.set_ylabel('Mean Accuracy', fontsize=11)
ax1.set_title('Accuracy vs Number of Approximated Layers', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Accuracy loss
ax2.bar(layer_impact['num_layers'], layer_impact['mean_loss'], 
       color='coral', alpha=0.7, label='Mean Loss')
ax2.plot(layer_impact['num_layers'], layer_impact['max_loss'], 
        marker='s', color='darkred', linewidth=2, markersize=6, label='Max Loss')
ax2.set_xlabel('Number of Approximated Layers', fontsize=11)
ax2.set_ylabel('Accuracy Loss', fontsize=11)
ax2.set_title('Accuracy Loss vs Number of Approximated Layers', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/Users/mabelwylie/Documents/toast-extensions/layer_approximation_impact.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved to: layer_approximation_impact.png")

## Export Sorted CSV for Further Analysis

In [ ]:
# Prepare comprehensive output sorted by model, dataset, accuracy
export_data = linear_results[[
    'model_short', 'dataset', 'seed', 'approx_layer', 'num_approx_layers',
    'accuracy', 'original_accuracy', 'delta_acc', 'params_saved_pct',
    'mlp_mode', 'attn_mode', 'head_dict', 'num_layers'
]].copy()

# Add computed columns
export_data['accuracy_retention_pct'] = (
    export_data['accuracy'] / export_data['original_accuracy'] * 100
).round(2)

# Sort by model, dataset, then accuracy (descending)
export_data = export_data.sort_values(
    by=['model_short', 'dataset', 'accuracy'],
    ascending=[True, True, False]
).reset_index(drop=True)

# Save to CSV
output_path = '/Users/mabelwylie/Documents/toast-extensions/linear_approximations_sorted.csv'
export_data.to_csv(output_path, index=False)

print(f"\nExported {len(export_data)} records to: {output_path}")
print(f"\nFirst 50 rows:")
print(export_data.head(50).to_string())

## Summary Statistics by Model

In [ ]:
model_summary = linear_results.groupby('model_short').agg({
    'accuracy': ['mean', 'std', 'min', 'max'],
    'delta_acc': ['mean', 'max'],
    'params_saved_pct': 'max',
    'original_accuracy': 'mean',
    'model_short': 'count'
}).round(4)

model_summary.columns = ['mean_acc', 'std_acc', 'min_acc', 'max_acc', 
                         'mean_loss', 'max_loss', 'max_params_saved', 'baseline_acc', 'num_experiments']

print("\nMODEL SUMMARY:")
print("="*120)
print(model_summary.to_string())

# Show models ranked by different criteria
print("\n\nRANKED BY BEST ACCURACY PRESERVATION:")
print(model_summary.sort_values('max_acc', ascending=False)[['mean_acc', 'min_acc', 'max_acc', 'max_loss']])

print("\n\nRANKED BY COMPRESSION POTENTIAL:")
print(model_summary.sort_values('max_params_saved', ascending=False)[['max_params_saved', 'mean_loss', 'max_loss']])

## Dataset-Specific Analysis

In [ ]:
dataset_summary = linear_results.groupby('dataset').agg({
    'accuracy': ['mean', 'std', 'min', 'max'],
    'delta_acc': ['mean', 'max'],
    'params_saved_pct': 'mean',
    'original_accuracy': 'mean'
}).round(4)

dataset_summary.columns = ['mean_acc', 'std_acc', 'min_acc', 'max_acc', 
                           'mean_loss', 'max_loss', 'avg_params_saved', 'baseline_acc']

print("\nDATASET SUMMARY:")
print("="*100)
print(dataset_summary.to_string())

# Show relative robustness
print("\n\nRELATIVE ROBUSTNESS (accuracy retention ratio):")
dataset_summary['robustness'] = (dataset_summary['mean_acc'] / dataset_summary['baseline_acc']).round(4)
print(dataset_summary[['baseline_acc', 'mean_acc', 'robustness', 'max_loss']].to_string())